[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent-memory.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239417-lesson-7-agent-with-memory)

# Agent 记忆

## 回顾

之前，我们构建了一个可以执行以下操作的 agent：

* `行动` - 让模型调用特定的工具
* `观察` - 将工具输出传回模型
* `推理` - 让模型对工具输出进行推理，以决定下一步要做什么（例如，调用另一个工具或直接响应）

![Screenshot 2024-08-21 at 12.45.32 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab7453080e6802cd1703_agent-memory1.png)

## 目标

现在，我们要通过引入记忆来扩展我们的 agent。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

我们将使用 [LangSmith](https://docs.smith.langchain.com/) 进行 [追踪](https://docs.smith.langchain.com/concepts/tracing)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

这遵循了我们之前的做法。

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """将 a 和 b 相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """将 a 和 b 相加。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """将 a 除以 b。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="你是一个有用的助手，负责对一组输入执行算术运算。")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些执行工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些决定控制流的移动方式
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# 显示
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

## 记忆

让我们像之前一样运行我们的 agent。

In [ ]:
messages = [HumanMessage(content="将 3 和 4 相加。")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

现在，让我们乘以 2！

In [ ]:
messages = [HumanMessage(content="将那个结果乘以 2。")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

我们没有保留初始聊天中 7 的记忆！

这是因为 [状态是瞬态的](https://github.com/langchain-ai/langgraph/discussions/352#discussioncomment-9291220)，仅限于单个图执行。

当然，这限制了我们进行有中断的多轮对话的能力。

我们可以使用 [持久化](https://langchain-ai.github.io/langgraph/how-tos/persistence/) 来解决这个问题！

LangGraph 可以使用检查点机制来在每一步后自动保存图状态。

这个内置的持久化层为我们提供了记忆功能，允许 LangGraph 从最后的状态更新开始继续。

最容易使用的检查点机制之一是 `MemorySaver`，它是图状态的内存键值存储。

我们只需要用检查点机制编译图，我们的图就有了记忆！

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
react_graph_memory = builder.compile(checkpointer=memory)

当我们使用记忆时，我们需要指定一个 `thread_id`。

这个 `thread_id` 将存储我们的图状态集合。

这里是一个图解：

* 检查点机制在图的每一步都写入状态
* 这些检查点保存在一个线程中
* 我们可以在将来使用 `thread_id` 访问该线程

![state.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e0e9f526b41a4ed9e2d28b_agent-memory2.png)

In [ ]:
# 指定一个线程
config = {"configurable": {"thread_id": "1"}}

# 指定输入
messages = [HumanMessage(content="将 3 和 4 相加。")]

# 运行
messages = react_graph_memory.invoke({"messages": messages},config)
for m in messages['messages']:
    m.pretty_print()

如果我们传递相同的 `thread_id`，那么我们可以从之前记录的状态检查点继续！

在这种情况下，上面的对话被捕获在线程中。

我们传递的 `HumanMessage`（`"将那个结果乘以 2。"`）被附加到上面的对话中。

所以，模型现在知道 `那个结果` 指的是 `3 和 4 的和是 7。`

In [ ]:
messages = [HumanMessage(content="将那个结果乘以 2。")]
messages = react_graph_memory.invoke({"messages": messages}, config)
for m in messages['messages']:
    m.pretty_print()

## LangGraph Studio


**⚠️ 免责声明**

自这些视频拍摄以来，我们已经更新了 Studio，使其可以在本地运行并在浏览器中打开。这现在是运行 Studio 的首选方式（而不是像视频中显示的使用桌面应用程序）。请参阅 [此处](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server) 关于本地开发服务器的文档和 [此处](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server) 的更多信息。要启动本地开发服务器，请在此模块的 `module-1/studio/` 目录中的终端中运行以下命令：

```
langgraph dev
```